# Notebook 1: Building the Ultimate Tic-Tac-Toe Game Engine

*This notebook is the companion to the essay series on AlphaZero for Ultimate Tic-Tac-Toe.  
The essays cover the mathematics; these notebooks cover the code.  
This first notebook builds the game engine — the foundation everything else will run on.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/uttt-alphazero/blob/main/notebooks/notebook_1_uttt_engine.ipynb)

---

## What we will build

By the end of this notebook you will have:
- A `UltimateTTT` class that fully implements the game rules
- A `render()` method that prints the board legibly
- A `to_tensor()` method that encodes the state for a neural network
- A `random_agent` and a `play_game()` loop
- A small tournament to verify the engine is balanced

No libraries beyond `numpy` are required for the engine itself.


## Imports

In [1]:
import numpy as np
import random
from typing import Optional, List, Tuple
from collections import Counter

np.random.seed(42)
random.seed(42)
print("numpy", np.__version__)


numpy 2.2.6


## 1. Board Representation

The board is a 9×9 numpy array — nine local 3×3 grids arranged in a larger 3×3 grid.

```
grid[r][c]
```

Row `r` and column `c` together tell us two things:
- **Which local board:** `lb_row = r // 3`, `lb_col = c // 3`
- **Which cell within it:** `cell_r = r % 3`, `cell_c = c % 3`

Cell values: `0` = empty, `1` = X (player 1), `2` = O (player 2).

We also track:
- `local_winners`: a 3×3 array recording who won each local board (`0` = ongoing, `1` = X, `2` = O, `-1` = draw)
- `active_board`: which local board the next player must play in (`None` on the first move)
- `current_player`: whose turn it is


In [2]:
class UltimateTTT:
    """
    Ultimate Tic-Tac-Toe game engine.

    Board layout (9x9 numpy array):
        grid[r][c]  ->  local board (r//3, c//3), cell (r%3, c%3)

    Cell values:
        0 = empty, 1 = X (player 1), 2 = O (player 2)

    local_winners[lb_r][lb_c]:
        0 = ongoing, 1 = X won, 2 = O won, -1 = draw
    """

    EMPTY = 0
    X     = 1
    O     = 2
    DRAW  = -1

    # All eight winning lines on a 3x3 grid
    WIN_LINES = [
        [(0,0),(0,1),(0,2)],  # top row
        [(1,0),(1,1),(1,2)],  # middle row
        [(2,0),(2,1),(2,2)],  # bottom row
        [(0,0),(1,0),(2,0)],  # left col
        [(0,1),(1,1),(2,1)],  # middle col
        [(0,2),(1,2),(2,2)],  # right col
        [(0,0),(1,1),(2,2)],  # diagonal
        [(0,2),(1,1),(2,0)],  # anti-diagonal
    ]

    def __init__(self):
        self.grid          = np.zeros((9, 9), dtype=np.int8)
        self.local_winners = np.zeros((3, 3), dtype=np.int8)
        self.active_board  = None   # None = first move, any board allowed
        self.current_player = self.X
        self.done          = False
        self.winner        = None   # set when done

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _local_board(self, lb_r: int, lb_c: int) -> np.ndarray:
        """Return a view of the 3x3 local board at (lb_r, lb_c)."""
        return self.grid[lb_r*3 : lb_r*3+3, lb_c*3 : lb_c*3+3]

    def _check_3x3(self, g: np.ndarray) -> int:
        """
        Check if a 3x3 grid has a winner.
        Returns: 1 (X wins), 2 (O wins), -1 (draw), 0 (ongoing).
        """
        for line in self.WIN_LINES:
            vals = [g[r, c] for r, c in line]
            if vals[0] != 0 and vals[0] == vals[1] == vals[2]:
                return vals[0]
        if np.all(g != 0):
            return self.DRAW
        return self.EMPTY

    # ------------------------------------------------------------------
    # Public interface
    # ------------------------------------------------------------------

    def get_legal_moves(self) -> List[Tuple[int, int]]:
        """Return list of legal (row, col) positions in the 9x9 grid."""
        if self.done:
            return []

        # Determine playable local boards
        if self.active_board is not None:
            lb_r, lb_c = self.active_board
            if self.local_winners[lb_r, lb_c] != 0:
                # Sent to a finished board: play anywhere open
                playable = [(r, c) for r in range(3) for c in range(3)
                            if self.local_winners[r, c] == 0]
            else:
                playable = [(lb_r, lb_c)]
        else:
            playable = [(r, c) for r in range(3) for c in range(3)
                        if self.local_winners[r, c] == 0]

        moves = []
        for lb_r, lb_c in playable:
            lb = self._local_board(lb_r, lb_c)
            for cr in range(3):
                for cc in range(3):
                    if lb[cr, cc] == 0:
                        moves.append((lb_r*3 + cr, lb_c*3 + cc))
        return moves

    def make_move(self, row: int, col: int) -> bool:
        """
        Place the current player's mark at (row, col).
        Returns True on success, False if the move is illegal.
        """
        if self.done or (row, col) not in self.get_legal_moves():
            return False

        # Place mark
        self.grid[row, col] = self.current_player

        # Identify local board and cell within it
        lb_r, lb_c = row // 3, col // 3
        cell_r, cell_c = row % 3, col % 3

        # Update local winner if this move decided the local board
        if self.local_winners[lb_r, lb_c] == 0:
            result = self._check_3x3(self._local_board(lb_r, lb_c))
            if result != 0:
                self.local_winners[lb_r, lb_c] = result

        # Check global winner
        global_result = self._check_3x3(self.local_winners)
        if global_result != 0:
            self.done   = True
            self.winner = global_result

        # Next active board = the cell position just played
        self.active_board = (cell_r, cell_c)

        # Switch player
        self.current_player = self.O if self.current_player == self.X else self.X
        return True

    def copy(self) -> 'UltimateTTT':
        """Deep copy of the game state (needed for MCTS tree search)."""
        g = UltimateTTT()
        g.grid           = self.grid.copy()
        g.local_winners  = self.local_winners.copy()
        g.active_board   = self.active_board
        g.current_player = self.current_player
        g.done           = self.done
        g.winner         = self.winner
        return g

print("UltimateTTT class defined.")


UltimateTTT class defined.


## 2. Visualising the Board

A good `render()` method is surprisingly important during development — it lets you quickly sanity-check that moves are being applied correctly.

We print the 9×9 grid with visual dividers between local boards, and highlight the active local board.


In [3]:
def render(game: UltimateTTT) -> str:
    """
    Pretty-print the 9x9 board with local-board dividers.

    Symbols: . = empty, X = player 1, O = player 2
    Local board status shown in the corner of each 3x3.
    """
    SYM = {0: '.', 1: 'X', 2: 'O'}
    WIN_SYM = {0: ' ', 1: 'X', 2: 'O', -1: '='}

    lines = []
    sep = '-------+-------+-------'

    for r in range(9):
        row_parts = []
        for lb_c in range(3):
            cells_str = ' '.join(SYM[game.grid[r, lb_c*3 + cc]] for cc in range(3))
            row_parts.append(cells_str)
        lines.append(' | '.join(row_parts))
        if r in (2, 5):
            lines.append(sep)

    # Annotate local board winners
    lines.append('')
    winner_rows = []
    for lb_r in range(3):
        row = ' '.join(
            f'[{WIN_SYM[game.local_winners[lb_r, lb_c]]}]'
            for lb_c in range(3)
        )
        winner_rows.append(row)
    lines.append('Local board status (. ongoing, X/O won, = draw):')
    lines.extend(winner_rows)

    lines.append('')
    if game.done:
        w = {1: 'X', 2: 'O', -1: 'Draw'}
        lines.append(f'Game over. Result: {w[game.winner]}')
    else:
        ab = game.active_board
        ab_str = f'({ab[0]}, {ab[1]})' if ab else 'any'
        lines.append(f"Active board: {ab_str}   Player to move: {'X' if game.current_player == 1 else 'O'}")

    return '\n'.join(lines)


# Quick test: make a few moves and render
game = UltimateTTT()
moves = [(4, 4), (4, 1), (1, 3), (3, 0), (0, 1)]
for m in moves:
    game.make_move(*m)

print(render(game))
print(f"\nLegal moves: {game.get_legal_moves()}")


. . . | . . . | . . .
. . . | . . . | . . .
. . . | . . . | . . .
-------+-------+-------
. . . | . . . | . . .
. . . | . X . | . . .
. . . | . . . | . . .
-------+-------+-------
. . . | . . . | . . .
. . . | . . . | . . .
. . . | . . . | . . .

Local board status (. ongoing, X/O won, = draw):
[ ] [ ] [ ]
[ ] [ ] [ ]
[ ] [ ] [ ]

Active board: (1, 1)   Player to move: O

Legal moves: [(3, 3), (3, 4), (3, 5), (4, 3), (4, 5), (5, 3), (5, 4), (5, 5)]


## 3. Win Detection

Let's step through how win detection works, so we can verify the logic.

`_check_3x3()` checks all eight winning lines of a 3×3 grid. It is called both on local boards (using the `grid` values) and on the global board (using `local_winners`).

The interesting edge case: a local board can end in a **draw** (all cells filled, no winner). In the global board, drawn local boards count as belonging to no one — neither player "owns" them for the purpose of winning the global game.


In [4]:
# Demonstrate win detection on an isolated 3x3 grid
from copy import deepcopy

test_grid = np.array([
    [1, 1, 1],
    [2, 2, 0],
    [0, 0, 0],
], dtype=np.int8)

g = UltimateTTT()
print("Row win (X):", g._check_3x3(test_grid))   # should be 1

test_grid2 = np.array([
    [1, 2, 1],
    [1, 2, 2],
    [2, 1, 2],
], dtype=np.int8)
print("Draw:       ", g._check_3x3(test_grid2))   # should be -1

test_grid3 = np.array([
    [1, 2, 0],
    [1, 2, 0],
    [0, 0, 0],
], dtype=np.int8)
print("Ongoing:    ", g._check_3x3(test_grid3))   # should be 0


Row win (X): 1
Draw:        -1
Ongoing:     0


## 4. Encoding State for a Neural Network

When we attach a neural network to this engine, it will need the board state as a **tensor** — a multi-dimensional array of floats.

We use three channels, each of shape (9, 9):

| Channel | Meaning |
|---------|---------|
| 0 | Positions of the current player's marks (1.0 where present) |
| 1 | Positions of the opponent's marks |
| 2 | Legal move mask (1.0 where a legal move exists) |

This encoding is always from the current player's perspective — channel 0 is always "my pieces," regardless of whether the current player is X or O. This symmetry helps the neural network generalise.

The full AlphaZero encoding also includes additional channels (e.g., the active board), but this three-channel version is a clean starting point.


In [5]:
def to_tensor(game: UltimateTTT) -> np.ndarray:
    """
    Encode game state as a (3, 9, 9) float32 tensor.

    Channel 0: current player's marks
    Channel 1: opponent's marks
    Channel 2: legal move mask
    """
    cp  = game.current_player
    opp = UltimateTTT.O if cp == UltimateTTT.X else UltimateTTT.X

    current_plane  = (game.grid == cp).astype(np.float32)
    opponent_plane = (game.grid == opp).astype(np.float32)

    legal_plane = np.zeros((9, 9), dtype=np.float32)
    for r, c in game.get_legal_moves():
        legal_plane[r, c] = 1.0

    return np.stack([current_plane, opponent_plane, legal_plane])  # shape (3, 9, 9)


# Demonstrate
game = UltimateTTT()
game.make_move(4, 4)  # X plays centre of centre board
t = to_tensor(game)
print("Tensor shape:", t.shape)
print("\nChannel 0 (current player = O, marks):")
print(t[0])
print("\nChannel 1 (opponent = X, marks):")
print(t[1])
print("\nChannel 2 (legal move mask):")
print(t[2])
print(f"\nLegal move count: {int(t[2].sum())}")


Tensor shape: (3, 9, 9)

Channel 0 (current player = O, marks):
[[0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Channel 1 (opponent = X, marks):
[[0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Channel 2 (legal move mask):
[[0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 1. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 1. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]]

Legal move count: 8


## 5. Random Agent and Game Simulation

Before we can train anything, we need a baseline: an agent that picks moves uniformly at random.

We also need a game runner that accepts two agents (callables that take a game state and return a move) and plays them against each other to completion.


In [6]:
def random_agent(game: UltimateTTT) -> Tuple[int, int]:
    """Choose a uniformly random legal move."""
    return random.choice(game.get_legal_moves())


def play_game(
    agent1=None,
    agent2=None,
    verbose: bool = False
) -> int:
    """
    Play a complete game.

    agent1 plays X, agent2 plays O.
    Each agent is a callable: (UltimateTTT) -> (row, col).
    Defaults to random_agent.

    Returns: 1 (X wins), 2 (O wins), -1 (draw).
    """
    if agent1 is None: agent1 = random_agent
    if agent2 is None: agent2 = random_agent

    game = UltimateTTT()
    agents = {UltimateTTT.X: agent1, UltimateTTT.O: agent2}

    move_count = 0
    while not game.done:
        move = agents[game.current_player](game)
        game.make_move(*move)
        move_count += 1

        if verbose:
            print(f"--- Move {move_count}: {move} ---")
            print(render(game))
            print()

    return game.winner


# Play one verbose game as a sanity check
print("=== Sample game (first 5 moves shown) ===\n")
game = UltimateTTT()
for i in range(5):
    move = random_agent(game)
    game.make_move(*move)
    print(f"Move {i+1}: {move}")
    print(render(game))
    print()


=== Sample game (first 5 moves shown) ===

Move 1: (1, 5)
. . . | . . . | . . .
. . . | . . X | . . .
. . . | . . . | . . .
-------+-------+-------
. . . | . . . | . . .
. . . | . . . | . . .
. . . | . . . | . . .
-------+-------+-------
. . . | . . . | . . .
. . . | . . . | . . .
. . . | . . . | . . .

Local board status (. ongoing, X/O won, = draw):
[ ] [ ] [ ]
[ ] [ ] [ ]
[ ] [ ] [ ]

Active board: (1, 2)   Player to move: O

Move 2: (3, 6)
. . . | . . . | . . .
. . . | . . X | . . .
. . . | . . . | . . .
-------+-------+-------
. . . | . . . | O . .
. . . | . . . | . . .
. . . | . . . | . . .
-------+-------+-------
. . . | . . . | . . .
. . . | . . . | . . .
. . . | . . . | . . .

Local board status (. ongoing, X/O won, = draw):
[ ] [ ] [ ]
[ ] [ ] [ ]
[ ] [ ] [ ]

Active board: (0, 0)   Player to move: X

Move 3: (1, 1)
. . . | . . . | . . .
. X . | . . X | . . .
. . . | . . . | . . .
-------+-------+-------
. . . | . . . | O . .
. . . | . . . | . . .
. . . | . . . | . . .
------

## 6. Running a Tournament

Let's run 10,000 random-vs-random games and check the win rates. A correctly implemented game should show X winning slightly more often (first-mover advantage), with a small draw rate.


In [7]:
def tournament(agent1=None, agent2=None, n_games: int = 10_000) -> dict:
    """
    Run a tournament and print win/draw rates.

    Returns dict with keys 1 (X wins), 2 (O wins), -1 (draws).
    """
    if agent1 is None: agent1 = random_agent
    if agent2 is None: agent2 = random_agent

    results = Counter()
    for _ in range(n_games):
        results[play_game(agent1, agent2)] += 1

    print(f"Tournament: {n_games:,} games")
    print(f"  X wins:  {results[1]:6,}  ({100*results[1]/n_games:5.1f}%)")
    print(f"  O wins:  {results[2]:6,}  ({100*results[2]/n_games:5.1f}%)")
    print(f"  Draws:   {results[-1]:6,}  ({100*results[-1]/n_games:5.1f}%)")
    return dict(results)


results = tournament(n_games=10_000)


Tournament: 10,000 games
  X wins:   4,147  ( 41.5%)
  O wins:   3,611  ( 36.1%)
  Draws:    2,242  ( 22.4%)


## 7. Game Length Distribution

How long do random games last? This tells us the typical number of moves, which affects how we think about the credit-assignment problem — how far back we need to propagate reward signals.


In [8]:
def game_length(agent1=None, agent2=None) -> int:
    """Return the number of moves in a completed game."""
    if agent1 is None: agent1 = random_agent
    if agent2 is None: agent2 = random_agent

    game = UltimateTTT()
    agents = {UltimateTTT.X: agent1, UltimateTTT.O: agent2}
    n = 0
    while not game.done:
        game.make_move(*agents[game.current_player](game))
        n += 1
    return n


lengths = [game_length() for _ in range(5_000)]
arr = np.array(lengths)
print(f"Game length over 5,000 random games:")
print(f"  Mean:   {arr.mean():.1f} moves")
print(f"  Median: {np.median(arr):.0f} moves")
print(f"  Min:    {arr.min()} moves")
print(f"  Max:    {arr.max()} moves")
print(f"  Std:    {arr.std():.1f}")


Game length over 5,000 random games:
  Mean:   58.9 moves
  Median: 59 moves
  Min:    31 moves
  Max:    78 moves
  Std:    6.6


---

## What's Next

This notebook built the game engine — the substrate on which everything else runs.

In **Notebook 2** we will implement Monte Carlo Tree Search: the selection, expansion, backup, and simulation loop that uses PUCT to navigate the 9^40-node game tree. The `copy()` and `get_legal_moves()` methods we built here are the only interface MCTS needs.

In **Notebook 3** we will define the neural network: a small ResNet that takes the (3, 9, 9) tensor from `to_tensor()` and produces a policy vector over 81 moves and a scalar value estimate.

The full self-play training loop — MCTS + neural network + experience replay + gradient updates — comes in Notebook 4.

---

*Key objects exported from this notebook:*
- `UltimateTTT` — game engine class
- `to_tensor(game)` — (3, 9, 9) float32 state encoding
- `render(game)` — human-readable board string
- `random_agent(game)` — baseline agent
- `play_game(agent1, agent2)` — full game runner
- `tournament(agent1, agent2, n_games)` — win-rate estimator
